#### **DPR**

In [1]:
%load_ext autoreload
%autoreload 2

Loading the dataset and creating dpr object

In [2]:
from src.data_utils.corpus import Corpus

# from src.retrievers.models.encoder import TransformerEncoder
from src.retrievers.dpr import DPR

from hydra import initialize, compose

with initialize(config_path="config", version_base=None):
    cfg = compose(config_name="config")

corpus = Corpus(dataset_name=cfg.dataset.name, base_dir=f'{cfg.base_path}/retrieval_data')

dpr = DPR(cfg, documents=corpus.all_documents)

/Users/rasmuslinnemann/Desktop/Courses/Semester7/02456 Deep Learning/Project/02456-retrieval-project/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
W1123 20:07:22.137000 52883 torch/distributed/elastic/multiprocessing/redirects.py:35] NOTE: Redirects are currently not supported in MacOs.
Loading weights: 100%|██████████| 199/199 [00:00<00:00, 5832.02it/s]
[transformers] BertModel LOAD REPORT from: google-bert/bert-base-uncased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.

In [3]:
corpus.ds

DatasetDict({
    train: Dataset({
        features: ['question', 'text', 'query_id', 'document_id'],
        num_rows: 5500
    })
    validation: Dataset({
        features: ['question', 'text', 'query_id', 'document_id'],
        num_rows: 500
    })
    test: Dataset({
        features: ['question', 'text', 'query_id', 'document_id'],
        num_rows: 648
    })
})

Loading the queries for all 3 splits, as in `main.py`

In [4]:
from src.data_utils.query import Query

### Prepare queries
train_queries = [Query(corpus.train['question'][idx], 
                        id=corpus.train['query_id'][idx], 
                        relevant_document_ids=corpus.relevant_ids[corpus.train['query_id'][idx]]
                        ) 
                    for idx in range(corpus.train.num_rows)
                ]

val_queries =   [Query(corpus.val['question'][idx], 
                        id=corpus.val['query_id'][idx], 
                        relevant_document_ids=corpus.relevant_ids[corpus.val['query_id'][idx]]
                        ) 
                    for idx in range(corpus.val.num_rows)
                ]

test_queries =  [Query(corpus.test['question'][idx], 
                        id=corpus.test['query_id'][idx], 
                        relevant_document_ids=corpus.relevant_ids[corpus.test['query_id'][idx]]
                        ) 
                    for idx in range(corpus.test.num_rows)
                ]

From each query, we can get the ground-truth documents using `.relevant_document_ids`.

In [5]:
print(train_queries[5].text)
print(train_queries[5].relevant_document_ids)

Hobby vs. Business
['doc7', 'doc8']


Now what we are missing is to vectorize both the query and all the documents in the corpus, so that we can compare the query vector to all the document vectors and retrieve the most similar documents.

* Since we are dealing with a small dataset, we don't need a full vector database, but we can use a standard numpy array.

In [6]:
# query    → query BERT → pool → q  (768)
# document → doc BERT   → pool → d  (768)
# score = q · d          ← dot product; high means relevant


We begin by embedding the query

* First step is to tokenize the sentence (the string is converted to numbers (e.g. tokens))

In [7]:
q_tokens = dpr.query_encoder.tokenizer(train_queries[5].text, return_tensors="pt").to(dpr.device)
print(q_tokens["input_ids"])
print(q_tokens["input_ids"].shape)

tensor([[  101, 17792,  5443,  1012,  2449,   102]], device='mps:0')
torch.Size([1, 6])


* Second step is to run the tokenized tensor through BERT which computes attention for each token against every other token and creates a vector embedding for each token

In [8]:
q_embedding = dpr.query_encoder.model(**q_tokens)
print(q_embedding.last_hidden_state.shape)

torch.Size([1, 6, 768])


* Since we only want one vector representing the query, we perform pooling.

In [9]:
q_pooled = dpr.query_encoder.pool(q_embedding.last_hidden_state, q_tokens['attention_mask'])
print(q_pooled.shape)

torch.Size([1, 768])


That is how we encode a query manually, but we have a function for it `encode_queries` which takes multiple queries as a batch.

If we want to embed the documents, we do more or less the exact same thing.

* Just for the sake of the example, we can embed the ground truth documents of q5, so we can expect a high similarity to the query

In [10]:
print(train_queries[5].id)
print(train_queries[5].relevant_document_ids)

q5
['doc7', 'doc8']


In [11]:
print(corpus.train[5]["text"][0]) #doc7
print(corpus.train[5]["text"][1]) #doc8

Miscellaneous income -- same category used for hobbies.
"You can list it as other income reported on line 21 of form 1040.  In TurboTax, enter at: - Federal Taxes tab (Personal in  Home & Business) - Wages & Income -“I’ll choose what I work on” Button Scroll down to: -Less Common Income -Misc Income, 1099-A, 1099-C. -The next screen will give you several choices. Choose ""Other reportable Income"". You will reach a screen where you can type a description of the income and the amount. Type in the amount of income and categorize as Tutoring."


Embedding doc7. This can also be done using `encode_documents`, if we want to embed multiple documents at once.

In [12]:
d_tokens = dpr.doc_encoder.tokenizer(corpus.train[5]["text"][0], return_tensors="pt").to(dpr.device)
print(d_tokens["input_ids"].shape)

d_embedding = dpr.doc_encoder.model(**d_tokens)
print(d_embedding.last_hidden_state.shape)

d_pooled = dpr.query_encoder.pool(d_embedding.last_hidden_state, d_tokens['attention_mask'])
print(d_pooled.shape)

torch.Size([1, 13])
torch.Size([1, 13, 768])
torch.Size([1, 768])


Now that we have both a query and a document, we can compute the similarity

In [13]:
q_pooled @ d_pooled.T

tensor([[45.1412]], device='mps:0', grad_fn=<MmBackward0>)

This similarity is what is computed in `query_document` too.

Now we can encode the corpus and perform a lookup before finetuning, to serve as a baseline.

* Note the embeddings are only saved in memory here. We can also save them to disk if we need to reuse them.
* Running `dpr.encode_corpus()` took me 8 minutes on MPS.

In [14]:
# dpr.encode_corpus()            # embeds all 17,110 documents, takes a while
# loads from cache
dpr.encode_corpus(cache_path='outputs/embeddings/fiqa_untrained_mean_512.pt')

In [15]:
dpr.doc_embeddings.shape

torch.Size([17110, 768])

In [16]:
results = dpr.Lookup([q.text for q in val_queries[:20]], k=3)
[[d.GetId() for d in docs] for docs in results]

[['doc16344', 'doc3288', 'doc1833'],
 ['doc12630', 'doc7996', 'doc6858'],
 ['doc10777', 'doc779', 'doc12630'],
 ['doc9851', 'doc8112', 'doc6858'],
 ['doc6858', 'doc3148', 'doc12356'],
 ['doc9851', 'doc6858', 'doc3117'],
 ['doc6858', 'doc2576', 'doc12226'],
 ['doc6814', 'doc14649', 'doc535'],
 ['doc2065', 'doc14465', 'doc4617'],
 ['doc17036', 'doc6858', 'doc355'],
 ['doc7254', 'doc5552', 'doc7996'],
 ['doc556', 'doc6096', 'doc1924'],
 ['doc10046', 'doc11913', 'doc11012'],
 ['doc6431', 'doc1924', 'doc10929'],
 ['doc1588', 'doc1586', 'doc4938'],
 ['doc4641', 'doc15217', 'doc6858'],
 ['doc8042', 'doc11949', 'doc9997'],
 ['doc11721', 'doc15343', 'doc12356'],
 ['doc8731', 'doc16344', 'doc4782'],
 ['doc16344', 'doc3304', 'doc9851']]

We could also try to score this on the recall.

In [17]:
from src.retrievers.utils import retrieveQueryAndGetRelevancies
from src.evaluation.metrics import Evaluate

k = cfg.top_k   # 10

for name, queries in [('train', train_queries), ('val', val_queries), ('test', test_queries)]:
    relevancies = retrieveQueryAndGetRelevancies(dpr, queries, k=k)
    print(f'{name}: {Evaluate(relevancies, queries)}')

train: {'recall': 0.018125180375180378, 'precision': 0.0044000000000000115, 'mrr': 0.016751731601731608}
val: {'recall': 0.010255555555555554, 'precision': 0.0026, 'mrr': 0.013485714285714285}
test: {'recall': 0.017649544385655493, 'precision': 0.004629629629629632, 'mrr': 0.01595446306094454}


Now we are ready to start training.

In [ ]:
# from src.training.loss import DPRLoss
# from src.training.batching import build_batch

# criterion = DPRLoss()
# q_texts, d_texts, pos_mask = build_batch(corpus.train[0:10])

# print(len(d_texts))                 # all positives of q0..q9
# print(pos_mask[5].nonzero())        # columns holding doc7 and doc8

# q = dpr.encode_queries(q_texts)
# d = dpr.encode_documents(d_texts)
# loss = criterion(q, d, pos_mask)
# loss

: 